In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from datetime import datetime, timedelta
import mysql.connector
import requests
from IPython.display import display

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="student",
    password="1234",
    database="autobusu_parkas"
)

print("Connected successfully")

Connected successfully


In [3]:
print("Database check:")

dbs = pd.read_sql("SHOW DATABASES;", conn)
display(dbs)

print("Tables in autobusu_parkas:")

tables = pd.read_sql("SHOW TABLES;", conn)
display(tables)

Database check:


,Database
0,autobusu_parkas
1,information_schema


Tables in autobusu_parkas:


,Tables_in_autobusu_parkas
0,routes
1,stopping_points
2,stops
3,timetables


In [4]:
routes = pd.read_sql("SELECT * FROM routes", conn)
timetables = pd.read_sql("SELECT * FROM timetables", conn)
stops = pd.read_sql("SELECT * FROM stops", conn)
stopping_points = pd.read_sql("SELECT * FROM stopping_points", conn)

display(routes)
display(timetables)
display(stops)
display(stopping_points)

,mid,pavadinimas,trukme,komentarai
0,M919,Vilnius – Kryžkalnis – Klaipėda,235,Via Elektrėnai


,tid,marsruto_id,savaites_diena,isvykimo_laikas,galioja_nuo,galioja_iki
0,123654,M919,1,0 days 07:50:00,2022-01-02,2026-12-30


,stid,pavadinimas,longitude,latitude
0,STOP001,Vilnius,25.2797,54.6872
1,STOP002,Elektrėnai,24.6630,54.7862
2,STOP003,Kryžkalnis,23.8298,55.4397
3,STOP004,Klaipėda,21.1443,55.7033


,sid,tvarkarascio_id,stoteles_id,sustojimo_nr,isvykimo_laikas
0,1,123654,STOP001,0,0 days 07:50:00
1,2,123654,STOP002,1,0 days 08:30:00
2,3,123654,STOP003,2,0 days 09:50:00
3,4,123654,STOP004,3,0 days 11:45:00


In [5]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 54.6872,
    "longitude": 25.2797,
    "hourly": "temperature_2m,precipitation"
}

response = requests.get(url, params=params)
weather = response.json()

print("Temperature:", weather["hourly"]["temperature_2m"][0])
print("Precipitation:", weather["hourly"]["precipitation"][0])

Temperature: 3.8
Precipitation: 0.0


In [6]:
departure = datetime(2026, 3, 31, 7, 50)

segments = [
    ("Vilnius", "Elektrėnai", 40, 0.8, "Heavy traffic"),
    ("Elektrėnai", "Kryžkalnis", 80, 0.9, "Road works"),
    ("Kryžkalnis", "Klaipėda", 115, 1.0, None)
]

In [7]:
time = departure
results = []

results.append({
    "stop": "Vilnius",
    "planned": "07:50",
    "predicted": "07:50",
    "delay": 0,
    "reason": "Start"
})

planned_times = {
    "Elektrėnai": departure + timedelta(minutes=40),
    "Kryžkalnis": departure + timedelta(minutes=120),
    "Klaipėda": departure + timedelta(minutes=235)
}

for seg in segments:
    duration = round(seg[2] / seg[3])
    time += timedelta(minutes=duration)

    delay = int((time - planned_times[seg[1]]).total_seconds() / 60)

    results.append({
        "stop": seg[1],
        "planned": planned_times[seg[1]].strftime("%H:%M"),
        "predicted": time.strftime("%H:%M"),
        "delay": delay,
        "reason": seg[4]
    })

df = pd.DataFrame(results)
display(df)

,stop,planned,predicted,delay,reason
0,Vilnius,07:50,07:50,0,Start
1,Elektrėnai,08:30,08:40,10,Heavy traffic
2,Kryžkalnis,09:50,10:09,19,Road works
3,Klaipėda,11:45,12:04,19,None


In [8]:
print("Max delay:", df["delay"].max())
print("Average delay:", df["delay"].mean())

Max delay: 19
Average delay: 12.0


In [9]:
print("FINAL CONCLUSION:\n")

print("The solution was implemented in Kali Linux using MariaDB, Python, pandas, and requests.")
print("A relational database 'autobusu_parkas' was created and used.")
print("Internal data were integrated with external API data.")
print("Predicted arrival times were calculated using segment duration and traffic conditions.")
print("The system supports periodic updates (simulated in the model).")
print("The maximum delay was 19 minutes and the average delay was 12 minutes.")

FINAL CONCLUSION:

The solution was implemented in Kali Linux using MariaDB, Python, pandas, and requests.
A relational database 'autobusu_parkas' was created and used.
Internal data were integrated with external API data.
Predicted arrival times were calculated using segment duration and traffic conditions.
The system supports periodic updates (simulated in the model).
The maximum delay was 19 minutes and the average delay was 12 minutes.


In [10]:
import pandas as pd
print("Database check:")

dbs = pd.read_sql("SHOW DATABASES;", conn)
display(dbs)

print("Tables in autobusu_parkas:")

tables = pd.read_sql("SHOW TABLES;", conn)
display(tables)

Database check:


,Database
0,autobusu_parkas
1,information_schema


Tables in autobusu_parkas:


,Tables_in_autobusu_parkas
0,routes
1,stopping_points
2,stops
3,timetables
